# Kaggle runner (thin)
All logic lives in the repo. This notebook only: pulls the repo, installs deps, runs **one** script.

Settings: Accelerator **GPU T4 x2**, Internet **On**. Add secrets (Add-ons -> Secrets) and set their names in the clone cell (default: `Hugging Face`, `mcp-git`).
To persist outputs: **Save Version** (Save & Run All), then use this notebook's output as a Dataset input in later runs.

In [ ]:
# ---- edit per run ----
REPO = "github.com/Arn15/mcp-poison-detect"
BRANCH = "main"
SCRIPT = "scripts/check_env.py"          # e.g. scripts/train_detector.py
ARGS = "--config configs/default.yaml"   # e.g. --set detector.use_context=false --tag b3
GPU_DEPS = False                          # True for attacker / agent-eval runs (installs vLLM)

In [ ]:
import os, subprocess, base64
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
def secret(k):
    try: return sec.get_secret(k)
    except Exception: return None
HF_SECRET, GH_SECRET = "Hugging Face", "mcp-git"   # names of your Kaggle secrets
os.environ["HF_TOKEN"] = secret(HF_SECRET) or ""
gh = secret(GH_SECRET)
WORK = "/kaggle/working"

# Token goes in an HTTP header for this command only: never in the URL, .git/config, or error output.
auth = []
if gh:
    b64 = base64.b64encode(f"x-access-token:{gh}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {b64}"]
def git(*args, cwd=None):
    r = subprocess.run(["git", *auth, *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(gh or "\0", "***")
        raise RuntimeError(f"git {args[0]} failed: {msg.strip()}")
    return r.stdout

os.chdir(WORK)
repo_dir = os.path.join(WORK, "mcp-poison-detect")
if os.path.isdir(os.path.join(repo_dir, ".git")):
    git("fetch", "-q", cwd=repo_dir); git("checkout", "-q", BRANCH, cwd=repo_dir); git("pull", "-q", cwd=repo_dir)
else:
    git("clone", "-q", "-b", BRANCH, f"https://{REPO}.git", repo_dir)
os.chdir(repo_dir)
print(git("log", "-1", "--oneline", cwd=repo_dir))

In [ ]:
!pip install -q -r requirements.txt
if GPU_DEPS:
    !pip install -q -r requirements-gpu.txt
!pip install -q -e . --no-deps

In [ ]:
# Resuming a previous run: attach its output as an input Dataset, then copy it in, e.g.
# !cp -r /kaggle/input/<prev-output>/outputs /kaggle/working/
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python {SCRIPT} {ARGS}